<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase55.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 55 — blinded external adjudication package and sealed scoring key

Phase 54 showed that the current authorized real-source family does not contain joinable companion evidence for the missing Arm A dimensions. Phase 55 therefore targets the other major publication gap: **independent blinded review**.

The notebook reconstructs the exact Phase 43 sealed holdout, freezes a balanced 60-case blinded adjudication subset, removes truth labels and detector outputs from the reviewer package, and stores the scoring key only in a private Drive workspace.

Creating the package is **not** itself independent validation. That status remains false until an eligible external reviewer completes the blinded response and a later phase scores it against the sealed key.


In [1]:
# CODE CELL 1/10 — Drive, constants, helpers
import os,re,csv,json,hashlib,sqlite3,shutil,tempfile,zipfile,time,random
import datetime as dt
from pathlib import Path
from copy import deepcopy
from collections import Counter,defaultdict

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='55_blinded_external_adjudication_package_and_sealed_scoring_key'

EXPECTED_P54_SHAREABLE='76a3dfa6f1c2ece69d4f89d8db298c8f9d78befde1399875ea6b05cf8b61293d'
EXPECTED_P39_DB='0c9507d8fcf5986e9763d473bf0b342a1c4daaad6c87ee04915986518f5c2384'
EXPECTED_P42_METHOD_FILE='490657f569f36fec6dd7c4b793556490a4c1318249fbb9111999f3150ab149a8'
EXPECTED_P42_METHOD_CONFIG='5bac736739cffa44b1ebbc2ced2886f7e267887a4e78a1b1ba8e012651616902'
EXPECTED_P43_SUMMARY='c24b174761f1e18e92c4d5abf761f7ff5d7c5f09740c8a7e3d0f63a9af50495f'
EXPECTED_P43_CASES='cf8f5c1c8db0b26ea524c746a536046f19b4b3414b5c2b5049efc53acc1cef7c'
EXPECTED_P43_EVENTS='f62992a75bbad73b36052327621d1176ff13bfd9887efc750feb80a4bff62c69'
EXPECTED_P43_LOCK='73a709b83dde4490aeac52ab35b150776e93499ac110a57e255e8cca2c459abc'
EXPECTED_TRUTH_SEAL='6952711786b4b99d37204dbbc70762caabfbdb852d0251be5ef231efaf72609b'

HOLDOUT_SEED=430930
BLIND_SAMPLE_SEED=550930

SAMPLE_PLAN={
 'CLEAN':6,
 'VALID_CM3_NORMALIZATION':6,
 'ANATOMY_LATERALITY_MISMATCH':4,
 'UNIT_MISSING_OR_INCOMPATIBLE':4,
 'VALUE_MISMATCH':4,
 'PERSON_LINK_MISMATCH':4,
 'STUDY_TIMEPOINT_LINK_MISMATCH':4,
 'MODEL_IDENTITY_LOSS':4,
 'PROVENANCE_LINK_LOSS':4,
 'TRACE_STATE_LOSS':4,
 'DUPLICATE_OBSERVATION':4,
 'DROPPED_OBSERVATION':4,
 'UNEXPECTED_OBSERVATION':4,
 'COMPOUND_HOLDOUT':4,
}
assert sum(SAMPLE_PLAN.values())==60

FIELDS=['obs_ref','person_ref','study_ref','biomarker_code','anatomy_source_code','laterality',
        'timepoint','effective_utc','value_mm3','unit_code','algorithm_name','algorithm_version',
        'trace_state','trace_version','device_ref','provenance_ref']

REASON_CODES=[
 'ANATOMY_LATERALITY_MISMATCH','UNIT_MISSING_OR_INCOMPATIBLE','VALUE_MISMATCH',
 'PERSON_LINK_MISMATCH','STUDY_TIMEPOINT_LINK_MISMATCH','MODEL_IDENTITY_LOSS',
 'PROVENANCE_LINK_LOSS','TRACE_STATE_LOSS','DUPLICATE_OBSERVATION',
 'DROPPED_OBSERVATION','UNEXPECTED_OBSERVATION'
]

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows:
            raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

if os.environ.get('PHASE55_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE55_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir():
        ROOTS.append(sd)

hint=os.environ.get('PHASE55_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute():
        hp=MYDRIVE/hp
    if not hp.is_dir():
        raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase55_'))
INPUT=WORK/'inputs'
INPUT.mkdir()
STAGE=WORK/'public_outputs'
STAGE.mkdir()

OUT_BASE=Path(os.environ.get('PHASE55_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase55')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')

PRIVATE_ROOT=MYDRIVE/'ICHI2027'/'_PRIVATE_EXTERNAL_VALIDATION'
PRIVATE_RUN=PRIVATE_ROOT/'Phase55'/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
PRIVATE_RUN.mkdir(parents=True,exist_ok=False)

print('Phase55 public output:',OUT)
print('Private sealed truth workspace prepared.')


Mounted at /content/drive
Phase55 public output: /content/drive/MyDrive/ICHI2027/Phase55/run_20261001T061113_823127Z
Private sealed truth workspace prepared.


In [2]:
# CODE CELL 2/10 — Auto-discover and checksum-verify exact Phase39/42/43/54 evidence

REQUIRED={
 'p54':('ICHI2027_Phase54_SHAREABLE','.json',EXPECTED_P54_SHAREABLE),
 'p39_db':('ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER','.sqlite',EXPECTED_P39_DB),
 'p42_method':('ICHI2027_Phase42_METHOD_SPEC','.json',EXPECTED_P42_METHOD_FILE),
 'p43_summary':('ICHI2027_Phase43_SHAREABLE','.json',EXPECTED_P43_SUMMARY),
 'p43_cases':('ICHI2027_Phase43_HOLDOUT_CASES','.csv',EXPECTED_P43_CASES),
 'p43_events':('ICHI2027_Phase43_HOLDOUT_EVENTS','.csv',EXPECTED_P43_EVENTS),
 'p43_lock':('ICHI2027_Phase43_METHOD_LOCK','.json',EXPECTED_P43_LOCK),
}

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

hits={k:[] for k in REQUIRED}
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            for k,(stem,ext,digest) in REQUIRED.items():
                if name_matches(fn,stem,ext):
                    p=Path(here)/fn
                    try:
                        if sha256(p)==digest:
                            hits[k].append(p)
                    except OSError:
                        pass

missing=[k for k,v in hits.items() if not v]
if missing:
    raise FileNotFoundError('Missing exact checksum-approved inputs: '+', '.join(missing))

P={k:sorted(v,key=str)[0] for k,v in hits.items()}

p54=json.loads(P['p54'].read_text('utf-8'))
p43=json.loads(P['p43_summary'].read_text('utf-8'))
lock43=json.loads(P['p43_lock'].read_text('utf-8'))
METHOD=json.loads(P['p42_method'].read_text('utf-8'))

assert p54['phase']=='54_authorized_companion_source_discovery_and_joinability_audit'
assert p54['dimensions_with_joinable_candidate_evidence']==0
assert p54['ready_for_official_armA'] is False

assert p43['phase']=='43_frozen_method_synthetic_holdout_no_retuning_audit'
assert p43['holdout_cases']==229
assert p43['holdout_authored_fault_events']==240
assert p43['method_retuned_after_holdout_generation'] is False
assert p43['independently_blinded_external_validation'] is False
assert lock43['holdout_truth_seal_sha256']==EXPECTED_TRUTH_SEAL
assert lock43['holdout_seed']==HOLDOUT_SEED

method_blob=json.dumps(METHOD,sort_keys=True,separators=(',',':')).encode('utf-8')
assert hashlib.sha256(method_blob).hexdigest()==EXPECTED_P42_METHOD_CONFIG

DB_LOCAL=INPUT/'immutable_source.sqlite'
shutil.copyfile(P['p39_db'],DB_LOCAL)
assert sha256(DB_LOCAL)==EXPECTED_P39_DB

print('PASS: exact Phase39/42/43/54 evidence chain verified.')


PASS: exact Phase39/42/43/54 evidence chain verified.


In [3]:
# CODE CELL 3/10 — Reconstruct exact Phase43 source reference

c=sqlite3.connect('file:'+str(DB_LOCAL)+'?mode=ro',uri=True)
assert c.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
assert c.execute('PRAGMA foreign_key_check').fetchall()==[]
c.row_factory=sqlite3.Row

STUDIES={r['study_ref']:dict(r) for r in c.execute('SELECT * FROM source_study')}
DEVICES={r['device_ref']:dict(r) for r in c.execute('SELECT * FROM source_device')}
PROVS={r['obs_ref']:dict(r) for r in c.execute('SELECT * FROM source_observation_provenance')}
SOURCE=[]

for raw in c.execute('SELECT * FROM source_observation ORDER BY obs_ref'):
    d=dict(raw)
    dev=DEVICES[d['device_ref']]
    d.update(algorithm_name=dev['algorithm_name'],algorithm_version=dev['algorithm_version'])
    assert PROVS[d['obs_ref']]['provenance_ref']==d['provenance_ref']
    SOURCE.append(d)
c.close()

assert len(SOURCE)==148 and len(STUDIES)==38 and len(DEVICES)==19

DDL=(
    "CREATE TABLE projected_study(study_ref TEXT,person_ref TEXT,study_started TEXT);\n"
    "CREATE TABLE projected_feature(obs_ref TEXT,person_ref TEXT,study_ref TEXT,biomarker_code TEXT,"
    "anatomy_source_code TEXT,laterality TEXT,timepoint TEXT,effective_utc TEXT,value_mm3 REAL,"
    "unit_code TEXT,algorithm_name TEXT,algorithm_version TEXT,trace_state TEXT,trace_version TEXT,"
    "device_ref TEXT,provenance_ref TEXT);\n"
)

INSERT='INSERT INTO projected_feature('+','.join(FIELDS)+') VALUES('+','.join('?' for _ in FIELDS)+')'

SQL_PROXY=(
    "WITH observed AS("
    " SELECT f.*,s.person_ref AS linked_person,s.study_started AS linked_started,"
    " CASE WHEN f.unit_code='mm3' THEN f.value_mm3 WHEN f.unit_code='cm3' THEN f.value_mm3*1000.0 ELSE NULL END AS normalized_mm3"
    " FROM projected_feature f LEFT JOIN projected_study s ON f.study_ref=s.study_ref"
    " WHERE f.biomarker_code='hippocampus-volume'"
    "),valid AS("
    " SELECT *,CASE WHEN linked_person=person_ref AND linked_started=effective_utc AND normalized_mm3>0"
    " AND provenance_ref IS NOT NULL"
    " AND ((laterality='LEFT' AND anatomy_source_code='SYN-HIP-LEFT')"
    " OR (laterality='RIGHT' AND anatomy_source_code='SYN-HIP-RIGHT'))"
    " AND timepoint IN ('baseline','12-month') THEN 1 ELSE 0 END AS valid_flag"
    " FROM observed"
    "),slots AS("
    " SELECT person_ref,timepoint,laterality,COUNT(*) AS n,SUM(valid_flag) AS nvalid"
    " FROM valid GROUP BY person_ref,timepoint,laterality"
    "),eligible AS("
    " SELECT person_ref FROM slots GROUP BY person_ref"
    " HAVING COUNT(*)=4 AND MIN(n)=1 AND MAX(n)=1 AND SUM(nvalid)=4"
    ") SELECT person_ref FROM eligible ORDER BY person_ref"
)

def query(rows,sql):
    cx=sqlite3.connect(':memory:')
    try:
        cx.executescript(DDL)
        cx.executemany('INSERT INTO projected_study VALUES (?,?,?)',
            [(s['study_ref'],s['person_ref'],s['study_started']) for s in STUDIES.values()])
        cx.executemany(INSERT,[tuple(r.get(k) for k in FIELDS) for r in rows])
        return {z[0] for z in cx.execute(sql)}
    finally:
        cx.close()

SOURCE_PROXY=query(SOURCE,SQL_PROXY)
ALL_PEOPLE=sorted({r['person_ref'] for r in SOURCE})
ELIGIBLE=sorted(SOURCE_PROXY)
HIP={p:[r['obs_ref'] for r in SOURCE
       if r['person_ref']==p and r['biomarker_code']=='hippocampus-volume']
     for p in ALL_PEOPLE}

assert len(ALL_PEOPLE)==19 and len(ELIGIBLE)==15
print('PASS: exact synthetic source reference reconstructed.')


PASS: exact synthetic source reference reconstructed.


In [4]:
# CODE CELL 4/10 — Reconstruct exact Phase43 holdout and verify pre-detection truth seal

RNG=random.Random(HOLDOUT_SEED)
HOLDOUT=[]
TRUTH={}
NORMS={}

def row_by_id(rows,obs):
    hit=[r for r in rows if r['obs_ref']==obs]
    assert len(hit)==1,obs
    return hit[0]

def new_case(kind,person):
    cid=f'holdout_{len(HOLDOUT):04d}'
    f={
        'case_id':cid,
        'kind':kind,
        'person_ref':person,
        'reference':deepcopy(SOURCE),
        'projected':deepcopy(SOURCE)
    }
    HOLDOUT.append(f)
    TRUTH[cid]=set()
    NORMS[cid]=set()
    return f

def mark(f,obs,reason):
    TRUTH[f['case_id']].add((obs,reason))

def inject(f,kind,obs):
    if kind=='ANATOMY_LATERALITY_MISMATCH':
        t=row_by_id(f['projected'],obs)
        t['anatomy_source_code']='SYN-HIP-RIGHT' if t['anatomy_source_code']=='SYN-HIP-LEFT' else 'SYN-HIP-LEFT'
        mark(f,obs,kind)
    elif kind=='UNIT_MISSING_OR_INCOMPATIBLE':
        row_by_id(f['projected'],obs)['unit_code']='mL'
        mark(f,obs,kind)
    elif kind=='VALUE_MISMATCH':
        t=row_by_id(f['projected'],obs)
        t['value_mm3']=float(t['value_mm3'])*1.01+1.0
        mark(f,obs,kind)
    elif kind=='PERSON_LINK_MISMATCH':
        t=row_by_id(f['projected'],obs)
        t['person_ref']=next(p for p in ALL_PEOPLE if p!=t['person_ref'])
        mark(f,obs,'PERSON_LINK_MISMATCH')
        mark(f,obs,'STUDY_TIMEPOINT_LINK_MISMATCH')
    elif kind=='STUDY_TIMEPOINT_LINK_MISMATCH':
        row_by_id(f['projected'],obs)['effective_utc']='2099-01-01T00:00:00Z'
        mark(f,obs,kind)
    elif kind=='MODEL_IDENTITY_LOSS':
        row_by_id(f['projected'],obs)['algorithm_name']=None
        mark(f,obs,kind)
    elif kind=='PROVENANCE_LINK_LOSS':
        row_by_id(f['projected'],obs)['device_ref']='Device/phase43-missing'
        mark(f,obs,kind)
    elif kind=='TRACE_STATE_LOSS':
        row_by_id(f['reference'],obs)['trace_state']='REVIEW_REQUIRED'
        row_by_id(f['projected'],obs)['trace_version']=None
        mark(f,obs,kind)
    elif kind=='DUPLICATE_OBSERVATION':
        f['projected'].append(deepcopy(row_by_id(f['projected'],obs)))
        mark(f,obs,kind)
    elif kind=='DROPPED_OBSERVATION':
        f['projected']=[r for r in f['projected'] if r['obs_ref']!=obs]
        mark(f,obs,kind)
    elif kind=='UNEXPECTED_OBSERVATION':
        t=deepcopy(row_by_id(f['projected'],obs))
        t['obs_ref']='Observation/phase43-unexpected-'+f['case_id']
        f['projected'].append(t)
        mark(f,t['obs_ref'],kind)
    else:
        raise AssertionError(kind)

for p in ALL_PEOPLE:
    new_case('CLEAN',p)

for p in ELIGIBLE:
    for fam in REASON_CODES:
        f=new_case(fam,p)
        inject(f,fam,RNG.choice(HIP[p]))

for p in ELIGIBLE:
    f=new_case('VALID_CM3_NORMALIZATION',p)
    obs=RNG.choice(HIP[p])
    t=row_by_id(f['projected'],obs)
    t['unit_code']='cm3'
    t['value_mm3']=float(t['value_mm3'])/1000.0
    NORMS[f['case_id']].add(obs)

for i in range(30):
    p=ELIGIBLE[i%len(ELIGIBLE)]
    obs1,obs2=RNG.sample(HIP[p],2)
    f=new_case('COMPOUND_HOLDOUT',p)
    if i%2==0:
        inject(f,'VALUE_MISMATCH',obs1)
        inject(f,'PROVENANCE_LINK_LOSS',obs2)
    else:
        inject(f,'UNIT_MISSING_OR_INCOMPATIBLE',obs1)
        inject(f,'DROPPED_OBSERVATION',obs2)

SEAL_ROWS=[
    {
        'case_id':f['case_id'],
        'kind':f['kind'],
        'person_ref':f['person_ref'],
        'truth_events':sorted([list(x) for x in TRUTH[f['case_id']]]),
        'expected_normalizations':sorted(NORMS[f['case_id']]),
        'projected_rows':len(f['projected'])
    }
    for f in HOLDOUT
]

seal=hashlib.sha256(
    json.dumps(SEAL_ROWS,sort_keys=True,separators=(',',':')).encode('utf-8')
).hexdigest()

assert len(HOLDOUT)==229
assert seal==EXPECTED_TRUTH_SEAL

with P['p43_cases'].open(newline='',encoding='utf-8') as f:
    cases43=list(csv.DictReader(f))
with P['p43_events'].open(newline='',encoding='utf-8') as f:
    events43=list(csv.DictReader(f))

assert len(cases43)==229 and len(events43)==240
assert {r['case_id'] for r in cases43}=={f['case_id'] for f in HOLDOUT}

print('PASS: exact Phase43 holdout reconstructed; truth seal verified before blinded sampling.')


PASS: exact Phase43 holdout reconstructed; truth seal verified before blinded sampling.


In [5]:
# CODE CELL 5/10 — Freeze balanced blinded sample

rng=random.Random(BLIND_SAMPLE_SEED)
by_kind=defaultdict(list)
for f in HOLDOUT:
    by_kind[f['kind']].append(f)

selected=[]
for kind,n in SAMPLE_PLAN.items():
    pool=by_kind[kind][:]
    rng.shuffle(pool)
    assert len(pool)>=n,(kind,len(pool),n)
    selected.extend(pool[:n])

rng.shuffle(selected)

assert len(selected)==60
assert len({f['case_id'] for f in selected})==60

BLIND_SALT=hashlib.sha256(
    f'{BLIND_SAMPLE_SEED}|{EXPECTED_TRUTH_SEAL}|phase55'.encode('utf-8')
).hexdigest()

def blind_case_id(case_id,index):
    digest=hashlib.sha256((BLIND_SALT+'|'+case_id).encode('utf-8')).hexdigest()[:10]
    return f'B{index:03d}-{digest}'

BLIND_MAP={}
for i,f in enumerate(selected,1):
    BLIND_MAP[f['case_id']]=blind_case_id(f['case_id'],i)

sample_lock={
    'project':PROJECT,
    'phase':PHASE,
    'holdout_truth_seal_sha256':EXPECTED_TRUTH_SEAL,
    'holdout_seed':HOLDOUT_SEED,
    'blind_sample_seed':BLIND_SAMPLE_SEED,
    'sample_size':len(selected),
    'sample_plan':SAMPLE_PLAN,
    'selected_original_case_ids':[f['case_id'] for f in selected],
    'blinded_case_ids':[BLIND_MAP[f['case_id']] for f in selected],
    'selection_kind':'BALANCED_ADJUDICATION_SAMPLE_NOT_PREVALENCE_SAMPLE'
}

sample_lock_sha=hashlib.sha256(
    json.dumps(sample_lock,sort_keys=True,separators=(',',':')).encode('utf-8')
).hexdigest()

print('PASS: 60-case balanced blinded sample frozen.')
print('Sample lock SHA-256:',sample_lock_sha)


PASS: 60-case balanced blinded sample frozen.
Sample lock SHA-256: b23e194c332a044f1d487f0b33c7cc7639b8031f1eb709f570ec02e97fca980e


In [6]:
# CODE CELL 6/10 — Build blinded source-vs-projected difference items

IDENTIFIER_FIELDS={'obs_ref','person_ref','study_ref','device_ref','provenance_ref'}

def mask_identifier(field,val):
    if val is None:
        return ''
    s=str(val)
    prefix={
        'obs_ref':'OBS',
        'person_ref':'P',
        'study_ref':'STUDY',
        'device_ref':'DEV',
        'provenance_ref':'PROV'
    }.get(field,'ID')
    digest=hashlib.sha256((BLIND_SALT+'|'+field+'|'+s).encode('utf-8')).hexdigest()[:8]
    return prefix+'_'+digest

def display_value(field,val):
    if val is None:
        return '<MISSING>'
    if field in IDENTIFIER_FIELDS:
        return mask_identifier(field,val)
    if isinstance(val,float):
        return format(val,'.8g')
    return str(val)

COMPARE_FIELDS=[
    'person_ref','study_ref','biomarker_code','anatomy_source_code','laterality',
    'timepoint','effective_utc','value_mm3','unit_code','algorithm_name',
    'algorithm_version','trace_state','trace_version','device_ref','provenance_ref'
]

def group_rows(rows):
    d=defaultdict(list)
    for r in rows:
        d[r['obs_ref']].append(r)
    return d

def build_diffs(f):
    ref=group_rows(f['reference'])
    proj=group_rows(f['projected'])
    items=[]

    for obs in sorted(set(ref)|set(proj)):
        rc=len(ref.get(obs,[]))
        pc=len(proj.get(obs,[]))
        obs_tok=mask_identifier('obs_ref',obs)

        if rc!=pc:
            items.append({
                'observation_token':obs_tok,
                'difference_type':'ROW_CARDINALITY',
                'field_name':'row_count',
                'source_value':str(rc),
                'projected_value':str(pc)
            })

        if rc==0 or pc==0:
            continue

        a=ref[obs][0]
        b=proj[obs][0]
        for field in COMPARE_FIELDS:
            if a.get(field)!=b.get(field):
                items.append({
                    'observation_token':obs_tok,
                    'difference_type':'FIELD_CHANGE',
                    'field_name':field,
                    'source_value':display_value(field,a.get(field)),
                    'projected_value':display_value(field,b.get(field))
                })

    if not items:
        items=[{
            'observation_token':'',
            'difference_type':'NO_DIFFERENCE',
            'field_name':'',
            'source_value':'',
            'projected_value':''
        }]
    return items

case_rows=[]
diff_rows=[]

for f in selected:
    bid=BLIND_MAP[f['case_id']]
    diffs=build_diffs(f)

    case_rows.append({
        'blinded_case_id':bid,
        'source_observation_rows':len(f['reference']),
        'projected_observation_rows':len(f['projected']),
        'difference_items':len(diffs),
        'truth_label_included':False,
        'method_output_included':False
    })

    for j,d in enumerate(diffs,1):
        diff_rows.append({
            'blinded_case_id':bid,
            'item_id':f'{bid}-D{j:02d}',
            **d
        })

assert len(case_rows)==60
assert all('kind' not in r and 'case_id' not in r for r in case_rows)
assert all('reason_code' not in r and 'detected' not in r for r in diff_rows)

print('PASS: blinded reviewer evidence built with no truth labels or detector outputs.')
print('Reviewer difference rows:',len(diff_rows))


PASS: blinded reviewer evidence built with no truth labels or detector outputs.
Reviewer difference rows: 74


In [7]:
# CODE CELL 7/10 — Create sealed private truth key

truth_rows=[]

for f in selected:
    cid=f['case_id']
    bid=BLIND_MAP[cid]
    reasons=sorted({reason for obs,reason in TRUTH[cid]})

    if reasons:
        truth_decision='FIDELITY_FAULT'
    elif NORMS[cid]:
        truth_decision='VALID_NORMALIZATION'
    else:
        truth_decision='NO_ISSUE'

    truth_rows.append({
        'blinded_case_id':bid,
        'original_case_id':cid,
        'original_kind':f['kind'],
        'truth_case_decision':truth_decision,
        'truth_reason_codes':'|'.join(reasons),
        'truth_event_count':len(TRUTH[cid]),
        'expected_normalization_count':len(NORMS[cid]),
        'original_person_ref':f['person_ref']
    })

TRUTH_KEY=PRIVATE_RUN/'ICHI2027_Phase55_PRIVATE_TRUTH_KEY.csv'
write_csv(TRUTH_KEY,truth_rows)

LOCK_PRIVATE=PRIVATE_RUN/'ICHI2027_Phase55_PRIVATE_SAMPLE_LOCK.json'
LOCK_PRIVATE.write_text(
    json.dumps({
        **sample_lock,
        'sample_lock_sha256':sample_lock_sha,
        'truth_key_sha256':sha256(TRUTH_KEY),
        'created_utc':dt.datetime.now(dt.timezone.utc).isoformat()
    },sort_keys=True,indent=2)+'\n',
    encoding='utf-8'
)

truth_distribution=Counter(r['original_kind'] for r in truth_rows)
assert truth_distribution==Counter(SAMPLE_PLAN)

print('PASS: private truth key and sample lock sealed.')
print('Private truth key SHA-256:',sha256(TRUTH_KEY))


PASS: private truth key and sample lock sealed.
Private truth key SHA-256: 7a090630dbc0e379952419f0afaf009a416e0527fdda5c42f77bf3b37e7b5a36


In [8]:
# CODE CELL 8/10 — Reviewer response template and instructions

response_rows=[
    {
        'blinded_case_id':r['blinded_case_id'],
        'reviewer_case_decision':'',
        'reviewer_reason_codes':'',
        'reviewer_confidence_1_to_5':'',
        'reviewer_notes':''
    }
    for r in case_rows
]

instruction_lines=[
    'ICHI 2027 Neuroimaging FHIR-to-OMOP Fidelity - Blinded External Adjudication',
    '',
    'PURPOSE',
    'You are reviewing a blinded balanced sample of synthetic source-vs-projected transformation cases.',
    'The fixture author labels and method outputs are hidden.',
    '',
    'INDEPENDENCE',
    'Please complete this review without consulting the project truth key, Phase 43 case labels, or method detector outputs.',
    'If you helped author the fixtures or tune the Phase 42 method, do not serve as the independent reviewer for this package.',
    '',
    'FOR EACH CASE',
    'Use ICHI2027_Phase55_BLINDED_CASE_SUMMARY.csv and ICHI2027_Phase55_BLINDED_DIFFERENCES.csv.',
    '',
    'Set reviewer_case_decision to exactly one of:',
    '  NO_ISSUE',
    '  VALID_NORMALIZATION',
    '  FIDELITY_FAULT',
    '  UNCERTAIN',
    '',
    'If FIDELITY_FAULT, enter one or more pipe-separated reason codes from:',
] + ['  '+x for x in REASON_CODES] + [
    '',
    'Reason-code meanings:',
    '  ANATOMY_LATERALITY_MISMATCH - anatomy, biomarker identity, or laterality is not preserved.',
    '  UNIT_MISSING_OR_INCOMPATIBLE - measurement unit is missing/incompatible rather than validly normalized.',
    '  VALUE_MISMATCH - numeric value is not equivalent after legitimate unit normalization.',
    '  PERSON_LINK_MISMATCH - evidence is linked to the wrong person.',
    '  STUDY_TIMEPOINT_LINK_MISMATCH - study/timepoint/date linkage is inconsistent.',
    '  MODEL_IDENTITY_LOSS - extraction algorithm/model identity or version is lost/changed.',
    '  PROVENANCE_LINK_LOSS - source provenance/device lineage is lost or broken.',
    '  TRACE_STATE_LOSS - reliability/review state or its version is lost/changed.',
    '  DUPLICATE_OBSERVATION - a source observation appears more than once in projected evidence.',
    '  DROPPED_OBSERVATION - a source observation is absent from projected evidence.',
    '  UNEXPECTED_OBSERVATION - an observation exists in projected evidence without a source counterpart.',
    '',
    'VALID NORMALIZATION',
    'A source mm3 value represented in cm3 with the mathematically equivalent numeric value is a valid normalization, not a fidelity fault.',
    '',
    'CONFIDENCE',
    'Use an integer from 1 (very uncertain) to 5 (very confident).',
    '',
    'RETURN',
    'Complete only ICHI2027_Phase55_REVIEWER_RESPONSE_TEMPLATE.csv.',
    'Do not rename blinded_case_id values and do not add/remove case rows.',
    '',
    'This package is balanced for adjudication and is NOT sampled to estimate real-world fault prevalence.'
]

instructions='\n'.join(instruction_lines)

inst_path=STAGE/'ICHI2027_Phase55_REVIEWER_INSTRUCTIONS.txt'
case_path=STAGE/'ICHI2027_Phase55_BLINDED_CASE_SUMMARY.csv'
diff_path=STAGE/'ICHI2027_Phase55_BLINDED_DIFFERENCES.csv'
resp_path=STAGE/'ICHI2027_Phase55_REVIEWER_RESPONSE_TEMPLATE.csv'

inst_path.write_text(instructions+'\n',encoding='utf-8')
write_csv(case_path,case_rows)
write_csv(diff_path,diff_rows)
write_csv(resp_path,response_rows)

print('PASS: reviewer-facing files created.')


PASS: reviewer-facing files created.


In [9]:
# CODE CELL 9/10 — Leakage audit of reviewer-facing package

forbidden_exact=set()
for r in truth_rows:
    forbidden_exact.add(r['original_case_id'])
    forbidden_exact.add(r['original_kind'])
    forbidden_exact.add(r['original_person_ref'])

review_data=case_path.read_text('utf-8')+'\n'+diff_path.read_text('utf-8')

leaks=[x for x in forbidden_exact if x and x in review_data]
assert not leaks,('Truth leakage in reviewer data',leaks[:10])

for term in [
    'sealed_truth','detected_truth','unexpected_alert','matched','missed',
    'false_alert','truth_seal_sha256'
]:
    assert term not in review_data

with resp_path.open(newline='',encoding='utf-8') as f:
    rr=list(csv.DictReader(f))

assert len(rr)==60
assert {r['blinded_case_id'] for r in rr}=={r['blinded_case_id'] for r in case_rows}

leakage_audit=[
    {'check':'original_case_ids_absent_from_reviewer_data','passed':True},
    {'check':'original_kind_labels_absent_from_reviewer_data','passed':True},
    {'check':'original_person_refs_absent_from_reviewer_data','passed':True},
    {'check':'truth_columns_absent_from_reviewer_data','passed':True},
    {'check':'detector_output_columns_absent_from_reviewer_data','passed':True},
    {'check':'response_template_has_exact_60_case_ids','passed':True},
    {'check':'truth_key_stored_private_only','passed':True}
]

print('PASS: blinded package leakage audit passed.')


PASS: blinded package leakage audit passed.


In [10]:
# CODE CELL 10/10 — Package reviewer ZIP + public audit summary

leak_path=STAGE/'ICHI2027_Phase55_BLINDING_LEAKAGE_AUDIT.csv'
write_csv(leak_path,leakage_audit)

distribution_public=[
    {'stratum':'NO_DIFFERENCE_CLEAN','cases':SAMPLE_PLAN['CLEAN']},
    {'stratum':'VALID_NORMALIZATION','cases':SAMPLE_PLAN['VALID_CM3_NORMALIZATION']},
    {'stratum':'SINGLE_FAULT_CASES_TOTAL','cases':sum(SAMPLE_PLAN[k] for k in REASON_CODES)},
    {'stratum':'COMPOUND_FAULT_CASES','cases':SAMPLE_PLAN['COMPOUND_HOLDOUT']},
    {'stratum':'TOTAL','cases':60}
]

dist_path=STAGE/'ICHI2027_Phase55_BLINDED_SAMPLE_DISTRIBUTION.csv'
write_csv(dist_path,distribution_public)

readme=STAGE/'ICHI2027_Phase55_README.txt'
readme.write_text(
    'Phase55 reconstructs the exact sealed Phase43 holdout and creates a balanced 60-case blinded external-adjudication package.\n'
    'The reviewer package excludes original case IDs, fixture kind labels, truth labels, detector outputs, and original synthetic person identifiers.\n'
    'The private truth key is stored only under the private external-validation workspace and is identified publicly only by SHA-256.\n'
    'The balanced sample is designed for blinded adjudication agreement, not for prevalence estimation.\n'
    'Independent validation remains FALSE until an eligible external reviewer completes the response template and a later phase scores it against the sealed key.\n',
    encoding='utf-8'
)

review_zip=STAGE/'ICHI2027_Phase55_EXTERNAL_REVIEW_PACKAGE.zip'
with zipfile.ZipFile(review_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in [inst_path,case_path,diff_path,resp_path]:
        z.write(p,p.name)

public_artifacts=[
    inst_path,case_path,diff_path,resp_path,leak_path,dist_path,readme,review_zip
]

summary={
    'project':PROJECT,
    'phase':PHASE,
    'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
    'input_phase54_shareable_sha256':sha256(P['p54']),
    'input_phase43_summary_sha256':sha256(P['p43_summary']),
    'input_phase43_cases_sha256':sha256(P['p43_cases']),
    'input_phase43_events_sha256':sha256(P['p43_events']),
    'input_phase43_method_lock_sha256':sha256(P['p43_lock']),
    'phase43_truth_seal_verified':True,
    'phase43_truth_seal_sha256':EXPECTED_TRUTH_SEAL,
    'phase42_method_config_sha256':EXPECTED_P42_METHOD_CONFIG,
    'holdout_reconstructed_cases':229,
    'blind_sample_seed':BLIND_SAMPLE_SEED,
    'blind_sample_size':60,
    'sample_selection_kind':'BALANCED_ADJUDICATION_SAMPLE_NOT_PREVALENCE_SAMPLE',
    'blind_sample_lock_sha256':sample_lock_sha,
    'private_truth_key_sha256':sha256(TRUTH_KEY),
    'reviewer_package_created':True,
    'reviewer_package_truth_labels_included':False,
    'reviewer_package_detector_outputs_included':False,
    'blinding_leakage_audit_passed':True,
    'external_reviewer_response_received':False,
    'independently_blinded_external_validation':False,
    'official_omop_etl_executed':False,
    'official_micdm_load_executed':False,
    'ready_for_official_armA':False,
    'next_step':'PHASE56_SCORE_COMPLETED_EXTERNAL_REVIEWER_RESPONSE_AGAINST_SEALED_TRUTH',
    'privacy':'SYNTHETIC_BLINDED_REVIEW_PACKAGE_PUBLIC_SAFE_PRIVATE_TRUTH_KEY_NOT_EXPORTED',
    'artifact_sha256':{p.name:sha256(p) for p in public_artifacts},
    'warning':'Package generation is preparation for independent blinded adjudication; it is not independent validation until an eligible external reviewer completes the response and the sealed scoring phase is executed.'
}

summary_path=STAGE/'ICHI2027_Phase55_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
public_artifacts.append(summary_path)

public_zip=STAGE/'ICHI2027_Phase55_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(public_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in public_artifacts:
        z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in public_artifacts+[public_zip]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2:
                raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase55 blinded external-review package frozen.')
print('Reviewer cases:',60)
print('Truth leakage audit: PASS')
print('Independent validation: FALSE pending external reviewer response')
print('Reviewer ZIP:',OUT/review_zip.name)
print('Shareable:',OUT/summary_path.name)


SUCCESS: Phase55 blinded external-review package frozen.
Reviewer cases: 60
Truth leakage audit: PASS
Independent validation: FALSE pending external reviewer response
Reviewer ZIP: /content/drive/MyDrive/ICHI2027/Phase55/run_20261001T061113_823127Z/ICHI2027_Phase55_EXTERNAL_REVIEW_PACKAGE.zip
Shareable: /content/drive/MyDrive/ICHI2027/Phase55/run_20261001T061113_823127Z/ICHI2027_Phase55_SHAREABLE.json


## After Phase 55

Send **only** `ICHI2027_Phase55_EXTERNAL_REVIEW_PACKAGE.zip` to an eligible independent reviewer.

Do **not** send the reviewer the Phase 43 case/event files, Phase 42 detector outputs, the private Phase 55 truth key, or anything that maps blinded IDs to original case IDs.

When the completed `ICHI2027_Phase55_REVIEWER_RESPONSE_TEMPLATE.csv` comes back, Phase 56 can score agreement against the already-sealed private truth key. Do not retune the method between package creation and scoring.
